# DRUGseqPy — Notebook 1: Quality Control

Covers the complete QC workflow:
1. Data loading & object construction
2. Metadata validation
3. Per-sample library QC (UMI, genes, %mito, %ribo, HK genes, Gini)
4. Plate-level QC (Z'-factor, SSMD, spatial heatmaps)
5. Group-level QC and replicate ICC
6. Normalization comparison via RLE plots
7. Sample and gene filtering (group-aware)

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

import drugseqpy as ds
from drugseqpy.utils import make_dummy_screen

sc.settings.verbosity = 1
plt.rcParams['figure.dpi'] = 100
print(f'drugseqpy v{ds.__version__}')

drugseqpy v0.1.0


## 1. Load data and create DrugSeqData

In [2]:
# Load your own data:
# counts = pd.read_csv('counts.csv', index_col=0)   # genes x samples
# obs    = pd.read_csv('metadata.csv', index_col=0) # samples x metadata

# Or generate synthetic data:
counts, obs = make_dummy_screen(
    n_genes=500, n_plates=2,
    n_dmso_per_plate=10, n_compounds=6, n_reps=4, seed=42
)
print(f'Counts: {counts.shape[0]} genes × {counts.shape[1]} samples')
obs.head()

Counts: 500 genes × 68 samples


,plate_id,compound,dose,dose_unit,sample_type,well_id
S0001,Plate01,DMSO,0.0,uM,DMSO,A01
S0002,Plate01,DMSO,0.0,uM,DMSO,B02
S0003,Plate01,DMSO,0.0,uM,DMSO,C03
S0004,Plate01,DMSO,0.0,uM,DMSO,A04
S0005,Plate01,DMSO,0.0,uM,DMSO,B05


In [5]:
# from drugseqpy.core import create_drugseq_object
# dsd = ds.create_drugseq_object(counts=counts, obs=obs)
dsd = create_drugseq_object(counts=counts, obs=obs)
print(dsd)

NameError: name 'create_drugseq_object' is not defined

## 2. Metadata validation

In [7]:
result = ds.validate_metadata(dsd)
print('Validation passed:', result['ok'])

AttributeError: module 'drugseqpy' has no attribute 'validate_metadata'

## 3. Per-sample QC metrics

Computes: total_umi, n_genes_det, pct_mito, pct_ribo, hk_cv, hk_dropout_frac, gini_index, outlier_score

In [ ]:
ds.compute_qc_metrics(dsd, mito_pattern='^MT-', ribo_pattern=r'^RP[SL]', inplace=True)
dsd.obs[['total_umi','n_genes_det','pct_mito','pct_ribo','hk_cv','gini_index','outlier_score']].describe()

In [ ]:
fig = ds.plot_qc_summary(
    dsd,
    metrics=['total_umi','n_genes_det','pct_mito','pct_ribo','hk_cv','gini_index','outlier_score'],
    group_by='plate_id',
    thresholds={'pct_mito': 20, 'n_genes_det': 200, 'outlier_score': 5},
)
plt.show()

In [ ]:
# Bivariate scatter: UMI vs detected genes
fig = ds.plot_qc_scatter(dsd, x_metric='log10_total_umi', y_metric='n_genes_det',
                          color_by='sample_type', y_threshold=200)
plt.show()

## 4. Plate-level QC

Z'-factor > 0.5 = excellent assay; spatial heatmaps reveal edge effects.

In [ ]:
ds.compute_plate_qc(dsd, signal_col='log10_total_umi', neg_ctrl_label='DMSO', inplace=True)
print(ds.plate_qc_summary(dsd)[['plate_id','zprime','ssmd','dmso_cv','dmso_cor','flag']])

In [ ]:
fig = ds.plot_zprime(dsd, show_ssmd=True)
plt.show()

In [ ]:
# Spatial heatmaps — one per plate
for pid in dsd.adata.uns['plate_qc']:
    fig = ds.plot_plate_heatmap(dsd, plate_id=pid, value_col='log10_total_umi')
    plt.title(f'Plate layout: {pid}')
    plt.show()

## 5. Group-level QC and replicate ICC

Mirrors macpie's `compute_qc_metrics()` group-level statistics.
ICC (intra-class correlation) > 0.8 = replicates are highly consistent.

In [ ]:
grp_qc = ds.compute_group_qc(dsd, group_by='compound', readout_col='total_umi')
print(grp_qc[['compound','group_median','sd_value','mad_value','z_score','cv_pct']].to_string())

In [ ]:
fig = ds.plot_group_qc_heatmap(grp_qc)
plt.show()

In [ ]:
icc = ds.compute_replicate_icc(dsd, group_by='compound', metric='total_umi')
print(f'\nICC = {icc["icc"]:.3f}  (> 0.8 indicates good replicate consistency)')
print(icc['per_group_cv'])

## 6. Select robust DMSO controls

Filters DMSO wells by TMMwsp Pearson correlation — mirrors macpie's `select_robust_controls()`.

In [ ]:
good_ctrl = ds.select_robust_controls(dsd, neg_ctrl_label='DMSO', min_cor=0.80)
print(f'Robust DMSO wells ({len(good_ctrl)}): {good_ctrl[:5]}...')

## 7. Normalization comparison

`limma_voom` typically achieves the lowest RLE deviation and CV for HTTr data
(validated by macpie benchmarks on MAC-seq data).

In [ ]:
# Side-by-side comparison of normalization methods
fig = ds.plot_norm_comparison(dsd, methods=['log1p','CPM','TMM','limma_voom'], subset_type='DMSO')
plt.show()

In [ ]:
# RLE plot for DMSO wells — boxes centered on 0 = good normalization
fig = ds.plot_rle(dsd, subset_type='DMSO', normalization='limma_voom',
                   label_col='well_id', color_col='plate_id')
plt.show()

## 8. Normalize, filter, and save

In [ ]:
# Apply limma_voom normalization (default, recommended)
ds.normalize_counts(dsd, method='limma_voom', inplace=True)

In [ ]:
# Filter low-quality samples
dsd_f = ds.filter_samples(dsd, min_umi=3000, max_pct_mito=20, max_outlier_score=8)
print(f'After sample filtering: {dsd_f.n_obs} samples remain')

In [ ]:
# Group-aware gene filtering (macpie::filter_genes_by_expression strategy)
# A gene is kept if expressed in >= 1 compound group, preventing removal
# of drug-induced genes that are off in DMSO controls.
dsd_f = ds.filter_genes(dsd_f, min_count=5, min_samples=2,
                          group_aware=True, group_col='compound')
print(f'After gene filtering: {dsd_f.n_vars} genes remain')

In [ ]:
# MDS for sample grouping (macpie::plot_mds approach)
# Better than PCA for initial QC: uses leading log-fold-change distances
fig = ds.plot_mds(dsd_f, group_by='sample_type', label_by='compound')
plt.show()

In [ ]:
# Save filtered object
dsd_f.adata.write_h5ad('dsd_qc_filtered.h5ad')
print('Saved: dsd_qc_filtered.h5ad')

---
**Next →** `02_Screen_Analysis.ipynb`